In [ ]:
import requests
import csv
import time
import statistics

HEADERS = {
    "User-Agent": "TiviMate/5.2.0 (Linux; Android 10)"
}

def check_stream_performance(url):
    try:
        start_time = time.perf_counter()
        r = requests.get(url, headers=HEADERS, timeout=3, stream=True)
        first_byte_time = time.perf_counter()
        response_ms = (first_byte_time - start_time) * 1000

        total_bytes = 0
        stream_start = time.perf_counter()
        for chunk in r.iter_content(chunk_size=1024*64):
            total_bytes += len(chunk)
            if time.perf_counter() - stream_start >= 1:
                break

        stream_elapsed = time.perf_counter() - stream_start
        load_mbps = (total_bytes / 1024 / 1024) / stream_elapsed

        if r.status_code == 200:
            return True, response_ms, load_mbps
        else:
            return False, None, None

    except Exception:
        return False, None, None


def test_multiple_times(url, attempts=2):
    responses = []
    loads = []
    success_count = 0

    for _ in range(attempts):
        ok, resp, load = check_stream_performance(url)

        if ok:
            success_count += 1
            responses.append(resp)
            loads.append(load)

        time.sleep(0.5)  # small delay between attempts

    if success_count == 0:
        return "OFFLINE", "N/A", "N/A", "N/A", "N/A"

    avg_resp = statistics.mean(responses)
    std_resp = statistics.stdev(responses) if len(responses) > 1 else 0

    avg_load = statistics.mean(loads)
    std_load = statistics.stdev(loads) if len(loads) > 1 else 0

    status = "ONLINE" if success_count > 0 else "OFFLINE"

    return (
        status,avg_resp,std_resp,avg_load,std_load
    )


input_file = "input.csv"
output_file = "output.csv"

with open(input_file, newline='', encoding='utf-8') as infile, \
     open(output_file, mode='w', newline='', encoding='utf-8') as outfile:

    reader = csv.DictReader(infile)

    fieldnames = [
        "host", "username", "password",
        "Status",
        "Avg Response (ms)", "Std Response",
        "Avg Load (MB/s)", "Std Load"
    ]

    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        host = row["host"].strip()
        username = row["username"].strip()
        password = row["password"].strip()

        url = f"{host}/{username}/{password}/20184"

        status, avg_resp, std_resp, avg_load, std_load = test_multiple_times(url)

        writer.writerow({
            "host": host,
            "username": username,
            "password": password,
            "Status": status,
            "Avg Response (ms)": avg_resp,
            "Std Response": std_resp,
            "Avg Load (MB/s)": avg_load,
            "Std Load": std_load
        })

        print(f"Checked: {url} -> {status}")

In [4]:
import pandas as pd

# Load CSV
df = pd.read_csv("output.csv")

# Clean column names (optional, helps avoid issues with spaces)
df.columns = df.columns.str.strip()

# Rename for easier handling (optional)
df = df.rename(columns={
    "Avg Response (ms)": "avg_response",
    "Avg Load (MB/s)": "avg_load"
})

# Group by host and compute mean values
grouped = df.groupby("host")[["avg_response", "avg_load"]].mean()

# Sort to compare easily
print("\n=== Sorted by Avg Response ===")
print(grouped.sort_values(by="avg_response"))

print("\n=== Sorted by Avg Load ===")
print(grouped.sort_values(by="avg_load"))

# Optional: direct comparison (correlation)
correlation = grouped["avg_response"].corr(grouped["avg_load"])
print(f"\nCorrelation between Avg Response and Avg Load: {correlation:.3f}")


=== Sorted by Avg Response ===
                             avg_response  avg_load
host                                               
http://tvmate.icu:8080         288.679867  1.993867
http://cord-cutter.net:8080    292.666104  1.794156
http://mytvstream.net:8080     297.040400  1.896000
http://xxip9.top:8080          302.621067  1.978000
http://1tv41.icu:8080          330.394800  1.428800

=== Sorted by Avg Load ===
                             avg_response  avg_load
host                                               
http://1tv41.icu:8080          330.394800  1.428800
http://cord-cutter.net:8080    292.666104  1.794156
http://mytvstream.net:8080     297.040400  1.896000
http://xxip9.top:8080          302.621067  1.978000
http://tvmate.icu:8080         288.679867  1.993867

Correlation between Avg Response and Avg Load: -0.878
